# FeatureLens v0.16 causal-position addendum

Use this notebook **only after the v0.15 full offline study completed**. It preserves that final-token causal baseline and runs the smaller max-feature-activation causal addendum.

It does **not** recollect the 224-prompt activation matrices, refit probes/features, rerun candidate stability, or rerun the 1/3/5 feature-set benchmark.


In [ ]:
# 1. Verify the Colab GPU runtime.
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime before continuing.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", torch.cuda.get_device_properties(0).total_memory / 1024**3)


In [ ]:
# 2. Mount Google Drive.
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# 3. Configuration — edit REPO_URL if needed.
from pathlib import Path

REPO_URL = "PASTE_YOUR_GIT_REPO_URL_HERE"
BRANCH = "main"
SOURCE_RUN_NAME = "FeatureLens_offline_v015"
ADDENDUM_RUN_NAME = "FeatureLens_offline_v016"

REPO_DIR = Path("/content/FeatureLens")
DRIVE_ROOT = Path("/content/drive/MyDrive")
SOURCE_ARTIFACTS = DRIVE_ROOT / SOURCE_RUN_NAME / "artifacts"
ADDENDUM_ROOT = DRIVE_ROOT / ADDENDUM_RUN_NAME
ADDENDUM_ARTIFACTS = ADDENDUM_ROOT / "artifacts"
LOG_PATH = ADDENDUM_ROOT / "causal_addendum.log"

if REPO_URL.startswith("PASTE_"):
    raise ValueError("Set REPO_URL to your FeatureLens Git repository URL first.")
if not SOURCE_ARTIFACTS.exists():
    raise FileNotFoundError(
        f"Could not find the completed v0.15 artifacts at {SOURCE_ARTIFACTS}. "
        "Change SOURCE_RUN_NAME if your previous Drive folder used another name."
    )
ADDENDUM_ARTIFACTS.mkdir(parents=True, exist_ok=True)
print("Source:", SOURCE_ARTIFACTS)
print("Addendum:", ADDENDUM_ARTIFACTS)


In [ ]:
# 4. Clone or refresh the v0.16 FeatureLens source.
import subprocess, shutil

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
print("Commit:", subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True).strip())


In [ ]:
# 5. Install runtime dependencies.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)


In [ ]:
# 6. Seed the addendum folder with only the small v0.15 study outputs.
# Large activation caches are intentionally NOT copied.
import shutil

for path in SOURCE_ARTIFACTS.rglob("*"):
    if not path.is_file():
        continue
    rel = path.relative_to(SOURCE_ARTIFACTS)
    if rel.parts and rel.parts[0] == "activations":
        continue
    if path.name.endswith((".complete", ".tmp")):
        continue
    destination = ADDENDUM_ARTIFACTS / rel
    if not destination.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, destination)

required = ["feature_catalog.csv", "layer_metrics.csv", "stability.csv", "selection_stability.csv", "feature_set_results.csv"]
missing = [name for name in required if not (ADDENDUM_ARTIFACTS / name).exists()]
if missing:
    raise RuntimeError(f"Previous study is missing required small artifacts: {missing}")
if not ((ADDENDUM_ARTIFACTS / "causal_results.csv").exists() or (ADDENDUM_ARTIFACTS / "causal_results_final_token.csv").exists()):
    raise RuntimeError("Previous study is missing its final-token causal baseline.")
print("Small v0.15 artifacts copied; large activations were skipped.")


In [ ]:
# 7. Point the repo's artifacts/ directory at the Drive-backed addendum folder.
import shutil
local_artifacts = REPO_DIR / "artifacts"
if local_artifacts.is_symlink():
    local_artifacts.unlink()
elif local_artifacts.exists():
    shutil.rmtree(local_artifacts)
local_artifacts.symlink_to(ADDENDUM_ARTIFACTS, target_is_directory=True)
print("artifacts ->", local_artifacts.resolve())


In [ ]:
# 8. Run only the max-active causal addendum + CPU synthesis.
# Task-level checkpointing makes this resumable if Colab disconnects.
import subprocess, sys, time

command = [sys.executable, "-m", "experiments.run_causal_addendum", "--resume"]
print("$", " ".join(command))
print("Log:", LOG_PATH)
start = time.time()
with LOG_PATH.open("a", encoding="utf-8") as log:
    log.write("\n\n=== FeatureLens v0.16 causal addendum ===\n")
    process = subprocess.Popen(command, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        log.write(line)
        log.flush()
    return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Addendum exited with code {return_code}. Fix the error and rerun this cell; --resume keeps completed causal tasks.")
print(f"\nCompleted in {(time.time()-start)/60:.1f} minutes.")


In [ ]:
# 9. Inspect the finalized position-sensitivity results.
import json, pandas as pd
from IPython.display import display, Markdown

position = pd.read_csv(ADDENDUM_ARTIFACTS / "causal_position_summary.csv")
study = pd.read_csv(ADDENDUM_ARTIFACTS / "study_feature_summary.csv")
summary = json.loads((ADDENDUM_ARTIFACTS / "study_summary.json").read_text(encoding="utf-8"))
report = (ADDENDUM_ARTIFACTS / "report.md").read_text(encoding="utf-8")

display(position[position["concept"] == "__all__"])
display(study)
display(summary)
display(Markdown(report))


In [ ]:
# 10. Create the final publishable result bundle.
import zipfile

PUBLISH_ZIP = ADDENDUM_ROOT / "FeatureLens_offline_results_v016.zip"
with zipfile.ZipFile(PUBLISH_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(ADDENDUM_ARTIFACTS.rglob("*")):
        if not path.is_file():
            continue
        rel = path.relative_to(ADDENDUM_ARTIFACTS)
        if rel.parts and rel.parts[0] == "activations":
            continue
        if path.name.endswith((".complete", ".tmp")):
            continue
        zf.write(path, arcname=str(Path("artifacts") / rel))
print("Final result bundle:", PUBLISH_ZIP)
print(f"Size: {PUBLISH_ZIP.stat().st_size / 1024**2:.2f} MiB")


## After the addendum

Download `FeatureLens_offline_results_v016.zip` and bring it back to the ChatGPT project before committing the empirical artifacts. The final public release should use the v0.16 report/summary rather than the older v0.15 headline.
